# Import thư viện sử dụng

Sử dụng 3 model:
 - lightgmb
 - catboost
 - xgboost

In [1]:
!pip install lightgbm xgboost catboost

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures, StandardScaler, Normalizer, LabelEncoder, OneHotEncoder
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import xgboost as xgb

# 1. Đọc dữ liệu

In [3]:

df = pd.read_csv('../data/preproces_data.csv')
df.head()

,Diện tích đất:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,Loại hình đất:,Chiều ngang:,Chiều dài:,Số phòng ngủ:,Số phòng vệ sinh:,Loại hình nhà ở:,...,Tầng số:,Hướng ban công:,Đặc điểm căn hộ:,Loại hình văn phòng:,district,city,Phân khu/Lô/Block/Tháp,has_floor_number,Price (trieu VND),log_price
0,170.0,Tây Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,13.0,13.00,2.0,2.0,"Nhà mặt phố, mặt tiền",...,26.0,Đông Bắc,Căn góc,Shophouse,thành phố tân an,long an,Không thuộc project/block,1,419.0,6.040255
1,64.2,Tây Bắc,Đã có sổ,Nở hậu,Đất thổ cư,4.0,16.05,2.0,2.0,"Nhà mặt phố, mặt tiền",...,26.0,Đông Bắc,Căn góc,Shophouse,quận 9,tp hồ chí minh,Không thuộc project/block,1,936.0,6.842683
2,108.0,Tây,Đã có sổ,Nở hậu,Đất thổ cư,5.0,20.00,6.0,6.0,"Nhà ngõ, hẻm",...,26.0,Đông Bắc,Căn góc,Shophouse,quận 7,tp hồ chí minh,Không thuộc project/block,1,9500.0,9.159152
3,42.0,Nam,Đã có sổ,Nở hậu,Đất thổ cư,4.0,11.00,5.0,3.0,"Nhà ngõ, hẻm",...,26.0,Đông Bắc,Căn góc,Shophouse,quận 11,tp hồ chí minh,Không thuộc project/block,1,6500.0,8.779711
4,93.0,Nam,Đã có sổ,Hẻm xe hơi,Đất thổ cư,5.5,17.00,5.0,6.0,"Nhà ngõ, hẻm",...,26.0,Đông Bắc,Căn góc,Shophouse,quận bình tân,tp hồ chí minh,Không thuộc project/block,1,5000.0,8.517393


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5947 entries, 0 to 5946
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Diện tích đất:            5947 non-null   float64
 1   Hướng cửa chính:          5947 non-null   object 
 2   Giấy tờ pháp lý:          5947 non-null   object 
 3   Đặc điểm nhà/đất:         5947 non-null   object 
 4   Loại hình đất:            5947 non-null   object 
 5   Chiều ngang:              5947 non-null   float64
 6   Chiều dài:                5947 non-null   float64
 7   Số phòng ngủ:             5947 non-null   float64
 8   Số phòng vệ sinh:         5947 non-null   float64
 9   Loại hình nhà ở:          5947 non-null   object 
 10  Tình trạng nội thất:      5947 non-null   object 
 11  Diện tích sử dụng:        5947 non-null   float64
 12  Tình trạng bất động sản:  5947 non-null   object 
 13  Diện tích:                5947 non-null   float64
 14  Loại hìn

# 1. One-hot endcoding với các 

In [5]:
object_cols = df.loc[:, df.dtypes == object].columns # tìm các cột kiểu dữ liệu object trong dataframe

In [6]:
df_temp = df.copy()
onehot_dict = {}

for col in object_cols:
    enc = OneHotEncoder(sparse_output=False)  # bỏ luôn .toarray()
    transformed = enc.fit_transform(df_temp[col].values.reshape(-1, 1))
    onehot_dict[col] = enc
    
    # Đặt tên cột theo tên cột gốc để tránh trùng
    feature_names = [f"{col}_{cat}" for cat in enc.categories_[0]]
    ohe_df = pd.DataFrame(transformed, columns=feature_names, index=df_temp.index)
    
    df_temp = df_temp.drop([col], axis=1)
    df_temp = df_temp.join(ohe_df)

In [7]:
df_temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5947 entries, 0 to 5946
Columns: 429 entries, Diện tích đất: to Phân khu/Lô/Block/Tháp_W3
dtypes: float64(428), int64(1)
memory usage: 19.5 MB


In [8]:
print("Số cột sau One-Hot:", df_temp.shape[1])

Số cột sau One-Hot: 429


Sau khi One-Hot Endcoding, tổng quan kết quả:
1) 429 columns: từ 29 columns ban đầu đã chuyển đổi thành 429 cột vì mỗi giá tị unique của cột chữ thành 1 cột riêng
2) 5947 mẫu dữ liệu (đáp ứng yêu cầu - tối thiểu 5000)
3) dtype gồm 2 loại: 
- float64 - 428 columns
- int64 - 1 columns

Tuy nhiên hiện tại các name column không phải ở dạng unidecode, nên XGBoost không xử lý được, cần chuyển về unidecode để phục vụ việc train data

In [9]:
from unidecode import unidecode
import re

def clean_col(col):
    col = unidecode(col)
    col = re.sub(r'[^A-Za-z0-9_]', '_', col)
    return col

df_temp.columns = [clean_col(col) for col in df_temp.columns]

print("Tên cột sau clean:", df_temp.columns.tolist()[:429])

Tên cột sau clean: ['Dien_tich_dat_', 'Chieu_ngang_', 'Chieu_dai_', 'So_phong_ngu_', 'So_phong_ve_sinh_', 'Dien_tich_su_dung_', 'Dien_tich_', 'Tong_so_tang_', 'Tang_so_', 'has_floor_number', 'Price__trieu_VND_', 'log_price', 'Huong_cua_chinh__Bac', 'Huong_cua_chinh__Nam', 'Huong_cua_chinh__Tay', 'Huong_cua_chinh__Tay_Bac', 'Huong_cua_chinh__Tay_Nam', 'Huong_cua_chinh__Dong', 'Huong_cua_chinh__Dong_Bac', 'Huong_cua_chinh__Dong_Nam', 'Giay_to_phap_ly__Giay_to_khac', 'Giay_to_phap_ly__Dang_cho_so', 'Giay_to_phap_ly__Da_co_so', 'Dac_diem_nha_dat__Hem_xe_hoi', 'Dac_diem_nha_dat__Mat_tien', 'Dac_diem_nha_dat__No_hau', 'Loai_hinh_dat__Dat_cong_nghiep', 'Loai_hinh_dat__Dat_nong_nghiep', 'Loai_hinh_dat__Dat_nen_du_an', 'Loai_hinh_dat__Dat_tho_cu', 'Loai_hinh_nha_o__Nha_biet_thu', 'Loai_hinh_nha_o__Nha_mat_pho__mat_tien', 'Loai_hinh_nha_o__Nha_ngo__hem', 'Loai_hinh_nha_o__Nha_pho_lien_ke', 'Tinh_trang_noi_that__Ban_giao_tho', 'Tinh_trang_noi_that__Hoan_thien_co_ban', 'Tinh_trang_noi_that__Noi_th

# 4. Chia tập train/test 

In [10]:
X = df_temp.drop(['Price__trieu_VND_','log_price'], axis=1)
Y = df_temp['log_price']

In [11]:
X

,Dien_tich_dat_,Chieu_ngang_,Chieu_dai_,So_phong_ngu_,So_phong_ve_sinh_,Dien_tich_su_dung_,Dien_tich_,Tong_so_tang_,Tang_so_,has_floor_number,...,Phan_khu_Lo_Block_Thap_S1,Phan_khu_Lo_Block_Thap_S4,Phan_khu_Lo_Block_Thap_S6,Phan_khu_Lo_Block_Thap_S7,Phan_khu_Lo_Block_Thap_SAI_GON_VILLAGE,Phan_khu_Lo_Block_Thap_T28,Phan_khu_Lo_Block_Thap_T5,Phan_khu_Lo_Block_Thap_THANG_LONG_CITY,Phan_khu_Lo_Block_Thap_TAN_KIEN_RESIDENCES,Phan_khu_Lo_Block_Thap_W3
0,170.00,13.0,13.00,2.0,2.0,40.0,103.0,1.0,26.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,64.20,4.0,16.05,2.0,2.0,40.0,103.0,1.0,26.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,108.00,5.0,20.00,6.0,6.0,226.0,103.0,2.0,26.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,42.00,4.0,11.00,5.0,3.0,226.0,103.0,2.0,26.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,93.00,5.5,17.00,5.0,6.0,226.0,103.0,5.0,26.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5942,33.00,6.0,11.00,4.0,3.0,31.0,35.0,4.0,18.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5943,56.00,4.0,14.50,3.0,2.0,96.0,35.0,4.0,18.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5944,68.00,4.0,17.00,3.0,2.0,96.0,35.0,4.0,18.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5945,68.00,4.0,17.00,1.0,1.0,96.0,36.0,4.0,18.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,Y, test_size = 0.3, random_state = 42
)

# 5. Huấn luyện mô hình

In [13]:
pipelines = {
    'linear':    [('model', LinearRegression())],
    'xgboost':   [('model', xgb.XGBRegressor(objective='reg:squarederror'))],
    'histogram': [('model', HistGradientBoostingRegressor())],
    'catboost':  [('model', CatBoostRegressor(verbose=0))],
    'lightGBM':  [('model', LGBMRegressor())]
}

In [14]:
results = []
for name, pipeline in pipelines.items():
    model = Pipeline(pipeline).fit(X_train.values, Y_train.values)
    Y_pred = model.predict(X_test.values)
    r2 = r2_score(Y_test, Y_pred)
    mse = mean_squared_error(Y_test, Y_pred)
    rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
    mae = mean_absolute_error(Y_test, Y_pred)
    results.append({
        'model': name,
        'r2': r2,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'model_train': model
    })

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001421 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1166
[LightGBM] [Info] Number of data points in the train set: 4162, number of used features: 150
[LightGBM] [Info] Start training from score 7.672900


/Users/nals_macbook_108/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [16]:
pd.DataFrame(results).sort_values(by=['r2', 'mse'],
                                  ascending=[False, True]).reset_index(drop=True)

,model,r2,mse,rmse,mae,model_train
0,catboost,0.437227,0.553492,0.743971,0.539087,"(CatBoostRegressor(loss_function='RMSE', verbo..."
1,lightGBM,0.415797,0.574569,0.758003,0.544672,(LGBMRegressor())
2,histogram,0.414926,0.575425,0.758568,0.543613,(HistGradientBoostingRegressor())
3,xgboost,0.414657,0.575690,0.758742,0.542116,"(XGBRegressor(base_score=None, booster=None, c..."
4,linear,-5.102234,6.001597,2.449816,0.649415,(LinearRegression())


### Metric đánh giá 
Metric r2: tỉ lệ phương sai dữ liệu mà mô hình giải thchs được
- r2 = 1.0 --> hoàn hảo, r2 = 0 --> không tốt hơn dự đoán trung bình, r2 < 0 --> tệ hơn dự đoán trung bình

Metric rmse/mse: sai số bình phương - phạt năng các lỗi lớn

Metric mae: sai số tuyệt đối trung bình - ít bị ảnh hưởng bởi outlier
### Phân tích chi tiết
- Nhóm gradient boosting (Mô hình catboost, lightGBM, XGboost, histogram)) hoạt động tốt với r2 ~ 0.41 -> 0.43, nghĩa là mô hình giải thích được > 41,4% ~ 43.7% biến động của dữ liệu, phù hợp với dữ liệu cấu trúc phi tuyến
- Linear Regression có r2 <0 - tệ, nguyên nhân:
+ dữ liệu chưa được chuẩn hoá (scaling) - Linear Regression rất nhạy cảm với scale features
+ dữ liệu có thể có ôutpier lớn ảnh hưởng nặng

--> Phân tích tiếp nhóm Gradient Boosting, tốt nhất sử dụng model Catboost làm mô hình chính vì r2 cao nhất và mae thấp nhất (53.9).